# Pipeline PCA + SVC — Imagens Intraorais Odontológicas

Pipeline:
1. Carregar sujeitos e dividir em **treino / validação / teste** no nível do sujeito (sem vazamento de dados)
2. Extrair o **canal de luminância (Y)** de cada imagem de treino
3. Achatar + **StandardScaler** + **PCA** — analisar a variância explicada por componente
4. Treinar um **SVC** com as features reduzidas pelo PCA
5. Avaliar nos conjuntos de validação e teste

In [ ]:
import random
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from PIL import Image
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.svm import SVC
from sklearn.metrics import classification_report, confusion_matrix

%matplotlib inline
print('Importações OK')

## 1. Configuração

In [ ]:
DATASET_ROOT = Path("/home/manolo/EngSoftware_UFPE/project/Dataset_CINUFPE_Odontologico")

# Redimensionar cada imagem antes de achatar — torna o PCA tratável
IMAGE_SIZE = (128, 128)

SEED        = 42
TRAIN_RATIO = 0.70
VAL_RATIO   = 0.15
TEST_RATIO  = 0.15

# Mapeamento do nome do arquivo → rótulo da classe
STEM_TO_LABEL = {
    "intraoral-frontal"           : "frontal",
    "intraoral-inferior"          : "inferior",
    "intraoral-superior"          : "superior",
    "intraoral-lateral-direita"   : "lateral_direita",
    "intraoral-lateral-esquerda"  : "lateral_esquerda",
}
CLASSES = list(STEM_TO_LABEL.values())

## 2. Divisão treino / validação / teste no nível do sujeito

In [ ]:
subject_dirs = sorted(p for p in DATASET_ROOT.iterdir() if p.is_dir())
n = len(subject_dirs)

rng = random.Random(SEED)
rng.shuffle(subject_dirs)

n_train = int(n * TRAIN_RATIO)
n_val   = int(n * VAL_RATIO)
# teste recebe o restante para nenhum sujeito ser perdido por arredondamento

train_subjects = subject_dirs[:n_train]
val_subjects   = subject_dirs[n_train : n_train + n_val]
test_subjects  = subject_dirs[n_train + n_val :]

print(f"Total de sujeitos : {n}")
print(f"  Treino          : {len(train_subjects)}")
print(f"  Validação       : {len(val_subjects)}")
print(f"  Teste           : {len(test_subjects)}")

## 3. Carregamento das imagens — apenas canal de luminância (Y)

In [ ]:
def load_luma(subjects, image_size=IMAGE_SIZE):
    """Retorna (X, y) onde X[i] é o canal de luminância achatado da imagem i."""
    X, y = [], []
    for subject in subjects:
        for img_path in sorted(subject.glob("*.jpeg")):
            if img_path.stem not in STEM_TO_LABEL:
                continue
            # YCbCr — Y é o componente de luminância (brilho)
            img_ycbcr = Image.open(img_path).convert("YCbCr")
            luma, _, _ = img_ycbcr.split()
            luma = luma.resize(image_size, Image.LANCZOS)
            X.append(np.array(luma, dtype=np.float32).ravel())
            y.append(STEM_TO_LABEL[img_path.stem])
    return np.array(X), np.array(y)


print("Carregando imagens de treino ...")
X_train_raw, y_train = load_luma(train_subjects)

print("Carregando imagens de validação ...")
X_val_raw, y_val = load_luma(val_subjects)

print("Carregando imagens de teste ...")
X_test_raw, y_test = load_luma(test_subjects)

print(f"\nX_train : {X_train_raw.shape}  — {len(np.unique(y_train))} classes")
print(f"X_val   : {X_val_raw.shape}")
print(f"X_test  : {X_test_raw.shape}")
print(f"Classes : {sorted(set(y_train))}")

### Exemplos de imagens de luminância (treino)

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(15, 3))
for ax, label in zip(axes, CLASSES):
    idx = np.where(y_train == label)[0][0]
    ax.imshow(X_train_raw[idx].reshape(IMAGE_SIZE), cmap='gray')
    ax.set_title(label, fontsize=9)
    ax.axis('off')
fig.suptitle("Uma imagem de luminância por classe (conjunto de treino)", y=1.02)
plt.tight_layout()
plt.show()

## 4. StandardScaler

Ajustado apenas no treino; validação e teste são transformados com os mesmos parâmetros.

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_raw)
X_val_scaled   = scaler.transform(X_val_raw)
X_test_scaled  = scaler.transform(X_test_raw)

print(f"Média das features após escalonamento (treino): {X_train_scaled.mean():.6f}")
print(f"Desvio padrão após escalonamento (treino)     : {X_train_scaled.std():.6f}")

## 5. PCA completo — distribuição de variabilidade por componente

In [ ]:
# PCA completo para analisar o espectro de variância
pca_full = PCA(random_state=SEED)
pca_full.fit(X_train_scaled)

evr        = pca_full.explained_variance_ratio_
cumulative = np.cumsum(evr)
n_total    = len(evr)

n95 = int(np.searchsorted(cumulative, 0.95)) + 1
n99 = int(np.searchsorted(cumulative, 0.99)) + 1

print(f"Total de componentes disponíveis : {n_total}")
print(f"Componentes para 95% de variância: {n95}")
print(f"Componentes para 99% de variância: {n99}")

In [ ]:
TOP_K = 100  # exibir os primeiros K componentes no gráfico de barras

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- variância individual explicada (primeiros TOP_K) ---
axes[0].bar(range(1, TOP_K + 1), evr[:TOP_K], color='steelblue', width=1.0)
axes[0].set_xlabel("Índice do componente")
axes[0].set_ylabel("Razão de variância explicada")
axes[0].set_title(f"Variância individual — primeiros {TOP_K} componentes")
axes[0].set_xlim([0, TOP_K + 1])

# --- variância explicada acumulada ---
axes[1].plot(range(1, n_total + 1), cumulative, color='steelblue', linewidth=1.5)
axes[1].axhline(0.95, color='red',    linestyle='--', label=f'95% → {n95} componentes')
axes[1].axhline(0.99, color='orange', linestyle='--', label=f'99% → {n99} componentes')
axes[1].axvline(n95,  color='red',    linestyle=':',  alpha=0.5)
axes[1].axvline(n99,  color='orange', linestyle=':',  alpha=0.5)
axes[1].set_xlabel("Número de componentes")
axes[1].set_ylabel("Variância explicada acumulada")
axes[1].set_title("Variância explicada acumulada")
axes[1].legend()
axes[1].set_xlim([0, n_total])
axes[1].set_ylim([0, 1.02])

plt.tight_layout()
plt.show()

### Top-20 eigenfaces (componentes principais visualizados)

In [ ]:
n_show = 20
fig, axes = plt.subplots(2, 10, figsize=(18, 4))
for i, ax in enumerate(axes.ravel()):
    if i >= n_show:
        ax.axis('off')
        continue
    componente = pca_full.components_[i].reshape(IMAGE_SIZE)
    ax.imshow(componente, cmap='RdBu_r')
    ax.set_title(f"CP{i+1}\n{evr[i]*100:.1f}%", fontsize=7)
    ax.axis('off')
fig.suptitle("Top-20 componentes principais (luminância)", y=1.01)
plt.tight_layout()
plt.show()

## 6. Redução com o número de componentes escolhido

In [ ]:
# Ajuste N_COMPONENTS para equilibrar acurácia e velocidade
N_COMPONENTS = n95

pca = PCA(n_components=N_COMPONENTS, random_state=SEED)
X_train_pca = pca.fit_transform(X_train_scaled)
X_val_pca   = pca.transform(X_val_scaled)
X_test_pca  = pca.transform(X_test_scaled)

print(f"N_COMPONENTS selecionado  : {N_COMPONENTS}")
print(f"Variância retida          : {pca.explained_variance_ratio_.sum():.4f}")
print(f"Forma de X_train_pca      : {X_train_pca.shape}")

## 7. Treinamento do SVC

In [ ]:
svc = SVC(kernel='rbf', C=10.0, gamma='scale', random_state=SEED, class_weight='balanced')
svc.fit(X_train_pca, y_train)

train_acc = svc.score(X_train_pca, y_train)
val_acc   = svc.score(X_val_pca,   y_val)

print(f"Acurácia no treino    : {train_acc:.4f}")
print(f"Acurácia na validação : {val_acc:.4f}")

## 8. Avaliação no conjunto de teste

In [ ]:
y_pred = svc.predict(X_test_pca)

print("=== Relatório de Classificação — Conjunto de Teste ===")
print(classification_report(y_test, y_pred, target_names=CLASSES))

In [ ]:
cm = confusion_matrix(y_test, y_pred, labels=CLASSES)

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(
    cm,
    annot=True, fmt='d',
    xticklabels=CLASSES, yticklabels=CLASSES,
    cmap='Blues', ax=ax
)
ax.set_xlabel("Previsto")
ax.set_ylabel("Verdadeiro")
ax.set_title(f"Matriz de Confusão — Conjunto de Teste (acurácia = {svc.score(X_test_pca, y_test):.4f})")
plt.tight_layout()
plt.show()